In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


DATA PRE_PROCESSING AND LOADING

In [ ]:
#loading the dataset
file_path = '/content/youtoxic_english_1000.csv'
yt_data = pd.read_csv(file_path)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
yt_data.head()

,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\nDont you reckon them 'black lives matter' ba...,True,True,False,False,True,False,False,False,False,False,False,False
3,Ugg7Gd006w1MPngCoAEC,04kJtp6pVXI,There are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,Ugg8FfTbbNF8IngCoAEC,04kJtp6pVXI,"The Arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
#Adding the labels to the dataset

def categorize_comment(row):
  hate_columns=['IsHatespeech','IsRacist','IsSexist','IsHomophobic','IsReligiousHate', 'IsRadicalism']
  offensive_columns = ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene']

  if row[hate_columns].any():
    return "hateful"
  elif row[offensive_columns].any():
    return "offensive"
  else:
    return "non-hateful"

yt_data['Labels'] = yt_data.apply(categorize_comment, axis = 1)

In [ ]:
yt_data.head()

,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism,Labels
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False,non-hateful
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False,offensive
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\nDont you reckon them 'black lives matter' ba...,True,True,False,False,True,False,False,False,False,False,False,False,offensive
3,Ugg7Gd006w1MPngCoAEC,04kJtp6pVXI,There are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False,non-hateful
4,Ugg8FfTbbNF8IngCoAEC,04kJtp6pVXI,"The Arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,False,False,non-hateful


In [ ]:
#cleaning the text in the dataset
import re
import nltk
from nltk.util import pr
from nltk.corpus import stopwords
import string

nltk.download('stopwords')
stemmer = nltk.SnowballStemmer("english")
stopword = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def clean(text):
    text = str(text).lower()  # Convert to lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'\@\w+|\#', '', text)  # Remove mentions and hashtags
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    text = re.sub(r'\n', '', text)  # Remove newlines
    text = re.sub(r'\w*\d\w*', '', text)  # Remove words containing digits
    text = [word for word in text.split() if word not in stopword]  # Remove stopwords
    text = " ".join(text)  # Join the words back into a single string
    return text

In [ ]:
yt_data_processed = yt_data[["Text","Labels"]]
yt_data_processed.head()

,Text,Labels
0,If only people would just take a step back and...,non-hateful
1,Law enforcement is not trained to shoot to app...,offensive
2,\nDont you reckon them 'black lives matter' ba...,offensive
3,There are a very large number of people who do...,non-hateful
4,"The Arab dude is absolutely right, he should h...",non-hateful


In [ ]:
yt_data_processed["Text"] = yt_data_processed["Text"].apply(clean)
yt_data_processed.head()

<ipython-input-10-13c3f8ad5c26>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yt_data_processed["Text"] = yt_data_processed["Text"].apply(clean)


,Text,Labels
0,people would take step back make case wasnt an...,non-hateful
1,law enforcement trained shoot apprehend traine...,offensive
2,dont reckon black lives matter banners held wh...,offensive
3,large number people like police officers calle...,non-hateful
4,arab dude absolutely right shot extra time sho...,non-hateful


In [ ]:
#splitting the dataset into test and train
X = np.array(yt_data_processed["Text"])
y = np.array(yt_data_processed["Labels"])
X[1]

# X = yt_data_processed["Text"]
# y = yt_data_processed["Labels"]
# X

'law enforcement trained shoot apprehend trained shoot kill thank wilson killing punk bitch'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
#converting the text into form suitable for training
cv = CountVectorizer()

#transforming the data
X_train_transformed = cv.fit_transform(X_train)
X_test_transformed = cv.transform(X_test)

Random Forest Classifier

In [ ]:
#Training the classifier
rf_model = RandomForestClassifier()
rf_model.fit(X_train_transformed, y_train)

RandomForestClassifier()

In [ ]:
#testing the classifier
y_pred_rf = rf_model.predict(X_test_transformed)

In [ ]:
accuracy_score_val=accuracy_score(y_test,y_pred_rf)
classification_report_val=classification_report(y_test,y_pred_rf)
confusion_matrix_val=confusion_matrix(y_test,y_pred_rf)

In [ ]:
print("Random Forest Results:")
print("Accuracy Score:", accuracy_score_val)
print("Classification Report:\n",classification_report_val)
print("Confusion matrix:\n",confusion_matrix_val)

Random Forest Results:
Accuracy Score: 0.59
Classification Report:
               precision    recall  f1-score   support

     hateful       0.50      0.05      0.10        55
 non-hateful       0.58      0.92      0.71       143
   offensive       0.63      0.42      0.51       102

    accuracy                           0.59       300
   macro avg       0.57      0.46      0.44       300
weighted avg       0.58      0.59      0.53       300

Confusion matrix:
 [[  3  39  13]
 [  0 131  12]
 [  3  56  43]]


In [ ]:
#Storing the model
import joblib
joblib.dump(rf_model,'random_forest_model.pkl')
joblib.dump(cv,'count_vectorizer.pkl')

['count_vectorizer.pkl']

In [ ]:
#Loading and testing the model
rf_model_loaded = joblib.load('random_forest_model.pkl')
cv_loaded =  joblib.load('count_vectorizer.pkl')


In [ ]:
#testing
test_text = "Stupid"
df = cv_loaded.transform(np.array([test_text]))
print(rf_model_loaded.predict(df))

['hateful']


Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train_transformed, y_train)

DecisionTreeClassifier()

In [ ]:
y_pred_dt = dt_model.predict(X_test_transformed)

In [ ]:
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))

Decision Tree Accuracy: 0.62
Decision Tree Classification Report:
               precision    recall  f1-score   support

     hateful       0.60      0.22      0.32        55
 non-hateful       0.64      0.83      0.72       143
   offensive       0.58      0.55      0.56       102

    accuracy                           0.62       300
   macro avg       0.61      0.53      0.54       300
weighted avg       0.61      0.62      0.60       300

Decision Tree Confusion Matrix:
 [[ 12  23  20]
 [  4 118  21]
 [  4  42  56]]


In [ ]:
import joblib
joblib.dump(dt_model,'decision_tree_model.pkl')

['decision_tree_model.pkl']

In [ ]:
#Loading and testing the model
dt_model_loaded = joblib.load('decision_tree_model.pkl')
cv_loaded =  joblib.load('count_vectorizer.pkl')

In [ ]:
#testing
test_text = "Worst"
df = cv_loaded.transform(np.array([test_text]))
print(dt_model_loaded.predict(df))

['non-hateful']


Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#Initaializing CountVectorizer for Naive Bayes
tfidf_nb = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [ ]:
X_train_tfidf = tfidf_nb.fit_transform(X_train)
X_test_tfidf = tfidf_nb.transform(X_test)

In [ ]:
# Train Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [ ]:
y_pred_nb = nb_model.predict(X_test_tfidf)

In [ ]:

accuracy = accuracy_score(y_test, y_pred_nb)
print("Naive Bayes Accuracy:", accuracy)
print("\nNaive Bayes Classification Report:\n", classification_report(y_test, y_pred_nb))
print("\nNaive Bayes Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.5066666666666667

Naive Bayes Classification Report:
               precision    recall  f1-score   support

     hateful       0.00      0.00      0.00        55
 non-hateful       0.49      0.99      0.66       143
   offensive       0.83      0.10      0.18       102

    accuracy                           0.51       300
   macro avg       0.44      0.36      0.28       300
weighted avg       0.52      0.51      0.37       300


Naive Bayes Confusion Matrix:
 [[  0  54   1]
 [  0 142   1]
 [  0  92  10]]


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
test_data = "you are a good"
test_data_tfidf = tfidf_nb.transform([test_data])
predicted_class = nb_model.predict(test_data_tfidf)

print(f"\nPredicted Label for new data: {predicted_class}")


Predicted Label for new data: ['non-hateful']


Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
label_mapping = {
    0: "hateful",
    1: "offensive",
    2: "non-hateful"
}

In [ ]:
def classify_into_classes(row):
  if row['Labels']=="hateful":
    return 0
  elif row['Labels']=="offensive":
    return 1
  else:
    return 2

yt_data_processed['class']=yt_data_processed.apply(classify_into_classes,axis=1)
yt_data_processed.head()

<ipython-input-39-6daeb22ad21b>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yt_data_processed['class']=yt_data_processed.apply(classify_into_classes,axis=1)


,Text,Labels,class
0,people would take step back make case wasnt an...,non-hateful,2
1,law enforcement trained shoot apprehend traine...,offensive,1
2,dont reckon black lives matter banners held wh...,offensive,1
3,large number people like police officers calle...,non-hateful,2
4,arab dude absolutely right shot extra time sho...,non-hateful,2


In [ ]:
#splitting the dataset into test and train
X = np.array(yt_data_processed["Text"])
y = np.array(yt_data_processed["class"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Use TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

In [ ]:
# Use Logistic Regression with class weights
logistic_model = LogisticRegression(class_weight=class_weight_dict, max_iter=1000)
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight={0: 2.8112449799196786, 1: 1.0510510510510511,
                                 2: 0.5907172995780591},
                   max_iter=1000)

In [ ]:
y_pred = logistic_model.predict(X_test_tfidf)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("Logistic Regression Classification Report:\n", classification_report(y_test, y_pred))

Logistic Regression Accuracy: 0.6566666666666666
Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.45      0.51        55
           1       0.69      0.57      0.62       102
           2       0.66      0.80      0.72       143

    accuracy                           0.66       300
   macro avg       0.64      0.61      0.62       300
weighted avg       0.66      0.66      0.65       300



In [ ]:
test_data = "you stupid"
test_data_transformed = tfidf.transform([test_data])
predicted_class = logistic_model.predict(test_data_transformed)

predicted_label = label_mapping[predicted_class[0]]

print(f"Predicted Class: {predicted_class[0]}")
print(f"Predicted Label: {predicted_label}")
predicted_class

Predicted Class: 2
Predicted Label: non-hateful


array([2])

Loading another dataset


In [ ]:
# from datasets import load_dataset

# ds = load_dataset("breadlicker45/youtube-comments-180k")

In [ ]:
# ds

In [ ]:
# df=ds['train'].to_pandas()
# df.head()